# Logistic Regression on the Titanic dataset

**Data source:** Kaggle — *Titanic: Machine Learning from Disaster* (Kaggle dataset). If you prefer, the same CSV is widely mirrored on GitHub and other public mirrors. 

**Notebook contents:**

- Download / load dataset (Kaggle API or fallback to seaborn / GitHub CSV)
- Data inspection & cleaning
- Feature engineering (encoding, simple interactions)
- Train / validation / test split
- Scaling where appropriate
- Logistic Regression with hyperparameter tuning (GridSearchCV)
- Evaluation: accuracy, ROC-AUC, confusion matrix, classification report
- Saving trained model

**Note:** this notebook includes two ways to obtain the data:

1. Using the Kaggle API (requires `kaggle` package and your Kaggle credentials configured).
2. If you don't have Kaggle credentials, the notebook falls back to loading Titanic from `seaborn` or a public GitHub CSV.

Replace the data-loading step with your preferred source if needed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

RANDOM_STATE = 42

## 1) Data download / load
The notebook will try three methods (in order):

- Kaggle API (`kaggle competitions download -c titanic`) — requires the `kaggle` package and configured credentials.
- A public GitHub raw CSV mirror (if Kaggle not available).
- `seaborn`'s built-in `titanic` dataset as a final fallback (note: seaborn's variant may have slightly different column names).

When running, pick the method that works for you. The Kaggle dataset page: (Kaggle — *Titanic*).

In [ ]:
# Option 1: try Kaggle API (uncomment to use)
# Requires: pip install kaggle and your ~/.kaggle/kaggle.json
# import os
# if not os.path.exists('data'):
#     os.makedirs('data')
# !kaggle competitions download -c titanic -p data
# After downloading, unzip and load train.csv/test.csv

# Option 2: try to read from a common GitHub mirror (uncomment to use)
# url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
# df = pd.read_csv(url)

# Option 3 (fallback): load seaborn's titanic (smaller and pre-cleaned differently)
try:
    df = sns.load_dataset('titanic')
    print('Loaded seaborn titanic dataset (fallback).')
except Exception as e:
    print('Failed to load seaborn titanic dataset:', e)
    df = pd.DataFrame()

# Show head
print('Shape:', df.shape)
df.head()

## 2) Basic inspection & cleaning
- Inspect missing values
- Select / rename columns to match the classic Kaggle layout if using seaborn fallback
- Simple imputation for Age and Embarked/Fare if necessary

In [ ]:
# Standardize column names if using seaborn fallback
if 'survived' in df.columns:
    # seaborn: 'survived' is already 0/1; Kaggle uses 'Survived'
    if 'Survived' not in df.columns:
        df = df.rename(columns={'survived':'Survived'})

# If columns differ, try to adapt
expected_cols = ['PassengerId','Survived','Pclass','Name','Sex','Age','SibSp','Parch','Ticket','Fare','Cabin','Embarked']
print('Columns present:', list(df.columns)[:20])

# Quick missing value overview
print('\nMissing values:')
print(df.isna().sum())

# If our dataframe lacks PassengerId (seaborn), create one
if 'PassengerId' not in df.columns:
    df['PassengerId'] = np.arange(1, len(df)+1)

# Keep only columns we need for this demo
cols_keep = ['PassengerId','Survived','Pclass','Name','Sex','Age','SibSp','Parch','Ticket','Fare','Cabin','Embarked']
for c in cols_keep:
    if c not in df.columns:
        df[c] = np.nan

df = df[cols_keep]

# Basic imputation examples
# Fill missing Embarked with mode, Fare with median, Age with median
if df['Embarked'].isna().any():
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode().iloc[0])
if df['Fare'].isna().any():
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
if df['Age'].isna().any():
    df['Age'] = df['Age'].fillna(df['Age'].median())

print('\nPost-imputation missing values:')
print(df.isna().sum())

df.head()

## 3) Feature engineering
- Convert Sex and Embarked to one-hot
- Create 'FamilySize' = SibSp + Parch + 1
- Extract Title from Name (Mr, Mrs, Miss, etc.) and group rare titles
- Drop columns unlikely to help (Ticket, Cabin -- or use them if you prefer)

In [ ]:
def extract_title(name):
    if pd.isna(name):
        return 'Unknown'
    import re
    m = re.search(r', (.*?)\.', name)
    if m:
        return m.group(1).strip()
    return 'Unknown'

# Feature engineering
df['FamilySize'] = df['SibSp'].fillna(0) + df['Parch'].fillna(0) + 1

if df['Name'].notna().any():
    df['Title'] = df['Name'].apply(extract_title)
    # Group rare titles
    rare_titles = df['Title'].value_counts()[df['Title'].value_counts()<10].index
    df.loc[df['Title'].isin(rare_titles), 'Title'] = 'Other'
else:
    df['Title'] = 'Unknown'

# Keep a concise set of features
features = ['Pclass','Sex','Age','Fare','Embarked','FamilySize','Title']
label = 'Survived'

df_model = df[features + [label]].copy()
print('Model dataset shape:', df_model.shape)
df_model.head()

## 4) Train / validation / test split
Split into train (70%), validation (15%), test (15%) with stratification on the label.

In [ ]:
# Drop rows with missing label
df_model = df_model.dropna(subset=[label])

X = df_model[features]
y = df_model[label].astype(int)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1764706, random_state=RANDOM_STATE, stratify=y_train_val)  # 0.17647 of 85% ≈ 15%

print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

## 5) Preprocessing pipeline
- Numeric features: Age, Fare, FamilySize -> impute (if needed) and scale
- Categorical features: Pclass, Sex, Embarked, Title -> OneHotEncode

We will build a `ColumnTransformer` inside a `Pipeline` with `LogisticRegression`.

In [ ]:
numeric_features = ['Age','Fare','FamilySize']
categorical_features = ['Pclass','Sex','Embarked','Title']

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Full pipeline with logistic regression
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('clf', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))])

# Quick fit to ensure pipeline runs
pipeline.fit(X_train.fillna({'Age':X_train['Age'].median(), 'Fare':X_train['Fare'].median(), 'FamilySize':1}), y_train)
print('Pipeline fit quick check OK')

## 6) Hyperparameter tuning (GridSearchCV)
Tune regularization strength `C` and solver. Use 5-fold CV on the training set.

In [ ]:
param_grid = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs', 'liblinear']
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)

grid.fit(X_train.fillna({'Age':X_train['Age'].median(), 'Fare':X_train['Fare'].median(), 'FamilySize':1}), y_train)

print('Best params:', grid.best_params_)
print('Best CV ROC-AUC:', grid.best_score_)

## 7) Evaluate on validation set

In [ ]:
best = grid.best_estimator_
X_val_prep = X_val.fillna({'Age':X_val['Age'].median(), 'Fare':X_val['Fare'].median(), 'FamilySize':1})
y_val_pred = best.predict(X_val_prep)
y_val_proba = best.predict_proba(X_val_prep)[:,1]

print('Validation accuracy:', accuracy_score(y_val, y_val_pred))
print('\nClassification report:\n', classification_report(y_val, y_val_pred))
print('\nConfusion matrix:\n', confusion_matrix(y_val, y_val_pred))
print('\nValidation ROC-AUC:', roc_auc_score(y_val, y_val_proba))

fpr, tpr, _ = roc_curve(y_val, y_val_proba)
plt.figure()
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.title('ROC curve (validation)')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.show()

## 8) Final evaluation on test set

In [ ]:
X_test_prep = X_test.fillna({'Age':X_test['Age'].median(), 'Fare':X_test['Fare'].median(), 'FamilySize':1})
y_test_pred = best.predict(X_test_prep)
y_test_proba = best.predict_proba(X_test_prep)[:,1]

print('Test accuracy:', accuracy_score(y_test, y_test_pred))
print('\nClassification report (test):\n', classification_report(y_test, y_test_pred))
print('\nConfusion matrix (test):\n', confusion_matrix(y_test, y_test_pred))
print('\nTest ROC-AUC:', roc_auc_score(y_test, y_test_proba))

## 9) Save model
Example of saving the trained pipeline with joblib.

In [ ]:
import joblib
model_path = 'logistic_titanic_pipeline.joblib'
joblib.dump(best, model_path)
print(f'Model saved to {model_path}')

## Data source
This notebook uses the Titanic dataset commonly provided on Kaggle: **"Titanic: Machine Learning from Disaster"**. Example Kaggle pages and public mirrors were consulted when preparing this notebook.